In [8]:
import requests
import pandas as pd
import json
import os

In [9]:
opt_code = ('')
                            # action(1/6)：从ifind super command勾选复制**同一到期日的合约**，粘贴到此
opt_code = (
    'HO2607-C-2550.CFE,AL2608C19600.SHF, HO2607-P-2550.CFE,AL2608C19600.SHF'
    )

underlying = '114.SH'    # action(2/6)：输入合约标的指数的代码

opt_date = '2026-07-01'     # action(3/6)：输入查询的日期，最好填最近一个交易日以获取最新的期权链

opt_EX = '2607'             # action(4/6)：输入合约月份

                            # action(5/6): 更新ifind token，刷新后access token有效期为7天
accessToken = '286c4713719c00645acce6dbe7f0b03bc10f4d8a.signs_ODgzMDE1NzQ5'

                            # action(6/6): 运行全部单元格，查看底部状态是否为“成功更新”

---
---


In [10]:
opt_var = ['ths_option_short_name_option']
start_date = opt_date
end_date = start_date
underlying_code = str(underlying + '_' + opt_EX)

In [11]:
RT_Url = 'https://quantapi.51ifind.com/api/v1/date_sequence'
thsHeaders = {"Content-Type":"application/json","access_token":accessToken}

thsPara_opt = {"codes":opt_code,
               'startdate':start_date,
               'enddate': end_date,
               'indipara': [{"indicator": x, "indiparams": []} for x in opt_var]
               }

thsResponse = requests.post(url=RT_Url, json=thsPara_opt, headers=thsHeaders)
data_opt = thsResponse.json()



In [12]:
rows = []
for item in data_opt["tables"]:
    thscode = item["thscode"]
    time = item["time"][0]  # time 是列表，取第一个元素
    name = item["table"]["ths_option_short_name_option"][0]  # 名称也是列表，取第一个

    # 拼成一行
    rows.append({
        "thscode": thscode,
        'date': time,
        'name': name
    })

df = pd.DataFrame(rows)

# 基础检查:上交所规定合约购/沽成对挂牌，df有偶数行
print(df)
if len(df) % 2 != 0:
    raise Exception("期权购/沽数不一致，请检查opt_code输入！")


Empty DataFrame
Columns: []
Index: []


In [13]:
# 可保存为其他格式的临时文件以便检查
# df.to_csv("thscode_name.csv", index=False, encoding="utf-8-sig")
# df.to_excel("thscode_name.xlsx", index=False)


In [14]:
file_path = "option_code_map.json"
# ==================================================
# 1. 从 df 生成最新的 code_to_name 字典
new_code_map = df.set_index("thscode")["name"].to_dict()

# 2. 如果文件存在，读取原有数据；不存在则创建空字典
if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        all_data = json.load(f)
else:
    all_data = {}

# 3. 【只更新目标合约，其他合约完全不动】
all_data[underlying_code] = {
    "code_to_name": new_code_map
}

# 4. 写回文件（覆盖整个文件，但只改了目标合约）
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"✅ 成功更新 {underlying_code} 类 code_to_name")

KeyError: "None of ['thscode'] are in the columns"